In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score

In [37]:
df = pd.read_csv("data/enriched_crop_data.csv")
df

,State,District,Season,Year,Crop,Area_Ha,Production_Ton,Yield_TonHa,Start_Year,District_norm,...,longitude,soil_ph,soil_oc,clay_pct,sand_pct,cec_cmol,avg_temp,humidity_avg,rain_total,solar_avg
0,Gujarat,Ahmadabad,Kharif,2015 - 2016,Arhar/Tur,1249.0,1535.0,1.23,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
1,Gujarat,Ahmadabad,Kharif,2015 - 2016,Bajra,755.0,882.0,1.17,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
2,Gujarat,Ahmadabad,Kharif,2015 - 2016,Castor seed,57920.0,86926.0,1.50,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
3,Gujarat,Ahmadabad,Kharif,2015 - 2016,Cotton(lint),131881.0,302016.0,2.29,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
4,Gujarat,Ahmadabad,Rabi,2015 - 2016,Gram,8087.0,5607.0,0.69,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,25.365137,29.049126,1.11,18.429180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75709,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Sweet potato,27.0,313.0,11.59,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033
75710,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Urad,2028.0,1168.0,0.58,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033
75711,Uttar Pradesh,Varanasi,Rabi,2022 - 2023,Wheat,70793.0,173726.0,2.45,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,20.513407,56.130055,122.69,14.440934
75712,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Sannhamp,377.0,221.0,0.59,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033


In [38]:
df.dropna(inplace=True)
df["Yield_qHa"] = df["Yield_TonHa"] * 10
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71417 entries, 0 to 75713
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State           71417 non-null  object 
 1   District        71417 non-null  object 
 2   Season          71417 non-null  object 
 3   Year            71417 non-null  object 
 4   Crop            71417 non-null  object 
 5   Area_Ha         71417 non-null  float64
 6   Production_Ton  71417 non-null  float64
 7   Yield_TonHa     71417 non-null  float64
 8   Start_Year      71417 non-null  int64  
 9   District_norm   71417 non-null  object 
 10  State_norm      71417 non-null  object 
 11  latitude        71417 non-null  float64
 12  longitude       71417 non-null  float64
 13  soil_ph         71417 non-null  float64
 14  soil_oc         71417 non-null  float64
 15  clay_pct        71417 non-null  float64
 16  sand_pct        71417 non-null  float64
 17  cec_cmol        71417 non-null  floa

In [39]:
np.array(df["Yield_qHa"]).mean()

np.float64(65.55013932257026)

In [40]:
cat_features = ["District", "Season", "Crop"]
num_features = [
    "Area_Ha",
    "latitude",
    "longitude",
    "soil_ph",
    "soil_oc",
    "clay_pct",
    "sand_pct",
    "cec_cmol",
    "avg_temp",
    "humidity_avg",
    "rain_total",
    "solar_avg",
]

df["Crop_Age"] = 2025 - df["Start_Year"]
num_features.append("Crop_Age")
df.drop(columns=["Start_Year"], inplace=True)

In [41]:
X = df.drop(columns=["Yield_qHa"])
y = df["Yield_qHa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((57133, 22), (14284, 22), (57133,), (14284,))

In [42]:
column_transformer = ColumnTransformer(
    [
        ("num", MinMaxScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

model_pipeline = Pipeline(
    [
        ("preprocessor", column_transformer),
        (
            "regressor",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=8,
            ),
        ),
    ]
)

In [43]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
y_pred = model_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

Mean Squared Error: 251.18
R^2 Score: 0.99
